In [3]:
import pandas as pd
df=pd.read_csv(r'C:\Users\Eshwar\OneDrive\Desktop\VS_Code\Swiggy\swiggy.csv')
display(df)

,id,name,city,rating,rating_count,cost,cuisine,lic_no,link,address,menu
0,567335,AB FOODS POINT,Abohar,--,Too Few Ratings,₹ 200,"Beverages,Pizzas",22122652000138,https://www.swiggy.com/restaurants/ab-foods-po...,"AB FOODS POINT, NEAR RISHI NARANG DENTAL CLINI...",Menu/567335.json
1,531342,Janta Sweet House,Abohar,4.4,50+ ratings,₹ 200,"Sweets,Bakery",12117201000112,https://www.swiggy.com/restaurants/janta-sweet...,"Janta Sweet House, Bazar No.9, Circullar Road,...",Menu/531342.json
2,158203,theka coffee desi,Abohar,3.8,100+ ratings,₹ 100,Beverages,22121652000190,https://www.swiggy.com/restaurants/theka-coffe...,"theka coffee desi, sahtiya sadan road city",Menu/158203.json
3,187912,Singh Hut,Abohar,3.7,20+ ratings,₹ 250,"Fast Food,Indian",22119652000167,https://www.swiggy.com/restaurants/singh-hut-n...,"Singh Hut, CIRCULAR ROAD NEAR NEHRU PARK ABOHAR",Menu/187912.json
4,543530,GRILL MASTERS,Abohar,--,Too Few Ratings,₹ 250,"Italian-American,Fast Food",12122201000053,https://www.swiggy.com/restaurants/grill-maste...,"GRILL MASTERS, ADA Heights, Abohar - Hanumanga...",Menu/543530.json
...,...,...,...,...,...,...,...,...,...,...,...
148536,553122,The Food Delight,Yavatmal,--,Too Few Ratings,₹ 200,"Fast Food,Snacks",21522053000452,https://www.swiggy.com/restaurants/the-food-de...,"The Food Delight, 94MC+X35, New Singhania Naga...",Menu/553122.json
148537,562647,MAITRI FOODS & BEVERAGES,Yavatmal,--,Too Few Ratings,₹ 300,Pizzas,license,https://www.swiggy.com/restaurants/maitri-food...,"MAITRI FOODS & BEVERAGES, POLIC MITRYA SOCIETY...",Menu/562647.json
148538,559435,Cafe Bella Ciao,Yavatmal,--,Too Few Ratings,₹ 300,"Fast Food,Snacks",21522251000378,https://www.swiggy.com/restaurants/cafe-bella-...,"Cafe Bella Ciao, SHOP NO 2 NEMANI MARKET SBI S...",Menu/559435.json
148539,418989,GRILL ZILLA,Yavatmal,--,Too Few Ratings,₹ 250,Continental,21521251000241,https://www.swiggy.com/restaurants/grill-zilla...,"GRILL ZILLA, SHO NO 2/6, POSTEL GROUND CHOWPAT...",Menu/418989.json


In [4]:
df_clean=df.drop_duplicates()
df_clean=df_clean.dropna()


In [ ]:
df_clean.duplicated().sum()
df_clean.head()

In [6]:
import pandas as pd

def split_city(city):
    if pd.isna(city) or str(city).strip() == "":
        return pd.Series(['NA', 'NA'])

    parts = str(city).split(',')

    sub_city = parts[0].strip()
    main_city = parts[1].strip() if len(parts) > 1 else parts[0].strip()

    return pd.Series([sub_city, main_city])

df_clean[['sub_city', 'main_city']] = df_clean['city'].apply(split_city)
df_clean.drop(columns=['city'], inplace=True)

print(df_clean.head())


       id               name rating     rating_count   cost  \
0  567335     AB FOODS POINT     --  Too Few Ratings  ₹ 200   
1  531342  Janta Sweet House    4.4      50+ ratings  ₹ 200   
2  158203  theka coffee desi    3.8     100+ ratings  ₹ 100   
3  187912          Singh Hut    3.7      20+ ratings  ₹ 250   
4  543530      GRILL MASTERS     --  Too Few Ratings  ₹ 250   

                      cuisine          lic_no  \
0            Beverages,Pizzas  22122652000138   
1               Sweets,Bakery  12117201000112   
2                   Beverages  22121652000190   
3            Fast Food,Indian  22119652000167   
4  Italian-American,Fast Food  12122201000053   

                                                link  \
0  https://www.swiggy.com/restaurants/ab-foods-po...   
1  https://www.swiggy.com/restaurants/janta-sweet...   
2  https://www.swiggy.com/restaurants/theka-coffe...   
3  https://www.swiggy.com/restaurants/singh-hut-n...   
4  https://www.swiggy.com/restaurants/grill-ma

In [7]:
TOP_ROWS = 10000
TARGET_CITIES = 10

rows_per_city = TOP_ROWS // TARGET_CITIES  # ~1666

df_top = (
    df_clean
    .sort_values('rating', ascending=False)   # keep better restaurants
    .groupby('main_city', group_keys=False)
    .head(rows_per_city)
    .reset_index(drop=True)
)

print("✅ Rows:", len(df_top))
print("✅ Cities:", df_top['main_city'].nunique())



✅ Rows: 68642
✅ Cities: 554


In [8]:
df_top['rating_count'] = df_top['rating_count'].replace({
    'Too Few Ratings': '10',
    '50+ ratings': '51',
    '100+ ratings': '101',
    '1K+ ratings': '1001',
    '5k ratings': '5001',
    '20+ ratings': '21',
    'nan': '0'  # If this is a string "nan", not actual NaN
})

In [9]:
def make_numeric_or_zero(value): #as few places as text instead of numbers
    return value if str(value).isnumeric() else 0

df_top['lic_no'] = df_top['lic_no'].apply(make_numeric_or_zero)


#name,cuisne,ratint
# Remove rupee symbol and convert to numeric
df_top['cost'] = df_top['cost'].replace('[₹,]', '', regex=True).astype(str)
df_top['cost'] = pd.to_numeric(df_top['cost'], errors='coerce').fillna(0).astype(int)


In [10]:
import pandas as pd


# Create copy for main_city
df_main = df_top.copy()
df_main["city"] = df_main["main_city"]

# Create copy for sub_city
df_sub = df_top.copy()
df_sub["city"] = df_sub["sub_city"]

# Combine both
df_expanded = pd.concat([df_main, df_sub], ignore_index=True)

# --- Keep only required columns ---
df_expanded = df_expanded[['name', 'rating', 'rating_count', 
                           'city', 'cuisine', 'cost', 'address']]

# Save expanded version
df_expanded.to_csv("cleaned_data.csv", index=False)

print(df_expanded.head())


               name rating rating_count       city               cuisine  \
0  Kurries & krolls    5.0           21     Mumbai   North Indian,Indian   
1  Creams and Bites    5.0           21  Bangalore             Ice Cream   
2   Begums Khansama    5.0           21      Delhi  Mughlai,North Indian   
3     House of Momo    5.0           21     Mumbai         Chinese,Asian   
4   Huber And Holly    5.0           21     Mumbai    Ice Cream,Desserts   

   cost                                            address  
0   250  Kurries & krolls, 101 Kailas Complex F Wing Pa...  
1   250  Creams and Bites, No 1, 1st floor, Sarjapura M...  
2   400  Begums Khansama, 160/9 pk bhatte wali gali, ki...  
3   350  House of Momo, Gala No. 28, Ground Floor, Swas...  
4   200  Huber And Holly, Gala No. 28, Ground Floor, Sw...  


In [ ]:
main city sub city ona append panitu one encoding panum

In [15]:
import pandas as pd
import pickle
from sklearn.preprocessing import OneHotEncoder
from scipy import sparse

# Load cleaned data
df = pd.read_csv("cleaned_data.csv")

# Columns to encode
cat_cols = ["city", "cuisine"]

# OneHotEncoder with sparse output (KEY FIX)
encoder = OneHotEncoder(
    sparse_output=True,
    handle_unknown="ignore",
    dtype="int8"
)

# Fit and transform
encoded_sparse = encoder.fit_transform(df[cat_cols])

# Save encoder
with open("encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

# Save sparse matrix (VERY memory efficient)
sparse.save_npz("encoded_features.npz", encoded_sparse)

# Save index mapping (important!)
df[['name', 'city', 'cuisine']].to_csv("index_mapping.csv", index=False)

print("✅ Sparse encoded features saved safely")


C:\Users\Eshwar\AppData\Local\Temp\ipykernel_19944\1300635407.py:7: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cleaned_data.csv")


✅ Sparse encoded features saved safely


In [16]:
import pandas as pd
from scipy import sparse
import pickle

# Load cleaned CSV
cleaned_df = pd.read_csv("cleaned_data.csv")

# Load sparse encoded features
encoded_sparse = sparse.load_npz("encoded_features.npz")

# Load encoder to get column names
with open("encoder.pkl", "rb") as f:
    encoder = pickle.load(f)

# Convert to DataFrame ONLY if dataset is small or filtered
encoded_df = pd.DataFrame(
    encoded_sparse.toarray(),
    columns=encoder.get_feature_names_out()
)

# Reset indices
cleaned_df = cleaned_df.reset_index(drop=True)
encoded_df = encoded_df.reset_index(drop=True)

# Optional: Save back (only if you really want CSV)
cleaned_df.to_csv("cleaned_data.csv", index=False)
encoded_df.to_csv("encoded_data.csv", index=False)

# Check indices
print("Indices match:", cleaned_df.index.equals(encoded_df.index))


C:\Users\Eshwar\AppData\Local\Temp\ipykernel_19944\1639638677.py:6: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  cleaned_df = pd.read_csv("cleaned_data.csv")


Indices match: True


In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
from scipy import sparse
import pickle

# -------------------------------
# 1️⃣ Load cleaned data
# -------------------------------
df = pd.read_csv("cleaned_data.csv")

# -------------------------------
# 2️⃣ Columns to use
# -------------------------------
categorical_cols = ['city', 'cuisine']
numeric_cols = ['rating', 'rating_count', 'cost']

# -------------------------------
# 3️⃣ Clean numeric columns
# -------------------------------
for col in numeric_cols:
    df[col] = df[col].replace(['--', '-', 'NA', 'N/A', 'na', 'null', 'None', ''], np.nan)
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# -------------------------------
# 4️⃣ Reduce categorical cardinality (memory safe)
# -------------------------------
for col in categorical_cols:
    top_vals = df[col].value_counts().nlargest(20).index
    df[col] = df[col].where(df[col].isin(top_vals), 'Other')

# -------------------------------
# 5️⃣ One-Hot Encoding (sparse)
# -------------------------------
encoder = OneHotEncoder(sparse_output=True, handle_unknown='ignore', dtype='int8')
encoded_cat_sparse = encoder.fit_transform(df[categorical_cols])

# -------------------------------
# 6️⃣ Combine numeric + encoded (sparse)
# -------------------------------
numeric_sparse = sparse.csr_matrix(df[numeric_cols].values)
df_encoded_sparse = sparse.hstack([numeric_sparse, encoded_cat_sparse]).tocsr()

# -------------------------------
# 7️⃣ Apply MiniBatchKMeans (memory safe)
# -------------------------------
kmeans = MiniBatchKMeans(n_clusters=5, random_state=42, batch_size=500)
clusters = kmeans.fit_predict(df_encoded_sparse)

# -------------------------------
# 8️⃣ Save encoded features + clusters
# -------------------------------
sparse.save_npz("df_encoded_features.npz", df_encoded_sparse)         # Sparse features
pd.DataFrame(clusters, columns=['Cluster']).to_csv("clusters.csv", index=False)  # Cluster labels

# -------------------------------
# 9️⃣ Save encoder for future use
# -------------------------------
with open("encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("✅ Clustering & encoding completed successfully")
print("📊 Cluster counts:")
print(pd.Series(clusters).value_counts())
print("📦 Sparse encoded features saved as df_encoded_features.npz")


C:\Users\Eshwar\AppData\Local\Temp\ipykernel_19944\1059490821.py:11: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cleaned_data.csv")


✅ Clustering & encoding completed successfully
📊 Cluster counts:
0    55678
3    46870
4    19098
2    12700
1     2938
Name: count, dtype: int64
📦 Sparse encoded features saved as df_encoded_features.npz


In [18]:
import pandas as pd
import numpy as np
from scipy import sparse

chunks = pd.read_csv("encoded_data.csv", chunksize=5000)

sparse_list = []

for chunk in chunks:

    # Convert all values to numeric
    chunk = chunk.apply(pd.to_numeric, errors='coerce')

    # Replace NaN with 0
    chunk = chunk.fillna(0)

    # Convert to float32 to reduce memory
    chunk = chunk.astype(np.float32)

    sparse_list.append(sparse.csr_matrix(chunk.values))

sparse_matrix = sparse.vstack(sparse_list)

# Save compressed matrix
sparse.save_npz("encoded_data.npz", sparse_matrix)

print("✅ NPZ file created successfully")

✅ NPZ file created successfully
